# 공정 단계별 전략 패턴 검정

정상 배치 90개를 각 배치의 진행률 기준으로 초기(0~20%), 성장기(20~50%), 생산기(50~80%), 후기(80~100%)로 나누고 RC·OC·APC 전략의 단계별 공정 특성을 비교한다. CSV는 저장하지 않는다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


def fdr_bh(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result


def welch_anova(*groups):
    groups = [np.asarray(group, dtype=float) for group in groups]
    sizes = np.array([len(group) for group in groups], dtype=float)
    means = np.array([group.mean() for group in groups])
    variances = np.array([group.var(ddof=1) for group in groups])
    weights = sizes / variances
    weight_sum = weights.sum()
    weighted_mean = np.sum(weights * means) / weight_sum
    k = len(groups)
    correction_sum = np.sum((1 - weights / weight_sum) ** 2 / (sizes - 1))
    numerator = np.sum(weights * (means - weighted_mean) ** 2) / (k - 1)
    denominator = 1 + 2 * (k - 2) * correction_sum / (k ** 2 - 1)
    statistic = numerator / denominator
    df2 = (k ** 2 - 1) / (3 * correction_sum)
    return statistic, stats.f.sf(statistic, k - 1, df2), df2


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
print(f'정상 배치 {data["배치번호"].nunique()}개')

### 확인

RC·OC·APC가 각각 30개 배치다. 그룹 분산이 같다고 강제하지 않도록 Welch ANOVA를 직접 계산하고, 28개 검정에는 FDR 보정을 적용한다.

In [ ]:
stage_ranges = {
    '초기': (0.0, 0.2), '성장기': (0.2, 0.5),
    '생산기': (0.5, 0.8), '후기': (0.8, 1.0000001),
}
rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)').copy()
    time = batch['발효시간(h)'].to_numpy()
    progress = time / time[-1]
    strategy = 'RC' if batch_number <= 30 else ('OC' if batch_number <= 60 else 'APC')
    for stage, (lower, upper) in stage_ranges.items():
        mask = (progress >= lower) & (progress < upper)
        stage_time = time[mask]
        penicillin = batch.loc[mask, '페니실린농도(g/L)'].to_numpy()
        rows.append({
            '배치번호': batch_number, '전략': strategy, '단계': stage,
            '페니실린기울기': np.polyfit(stage_time, penicillin, 1)[0],
            'pH표준편차': batch.loc[mask, 'pH'].std(ddof=1),
            '기질평균': batch.loc[mask, '기질농도(g/L)'].mean(),
            'OUR평균': batch.loc[mask, '산소소모율(g/min)'].mean(),
            'CO2평균': batch.loc[mask, '배가스이산화탄소(%)'].mean(),
            'DO평균': batch.loc[mask, '용존산소(mg/L)'].mean(),
            '온도표준편차': batch.loc[mask, '발효온도(K)'].std(ddof=1),
        })
stage_metrics = pd.DataFrame(rows)
display(stage_metrics.groupby(['단계', '전략']).size().unstack())

### 확인

각 단계·전략 조합에 배치 30개가 들어간다. 원시 시계열 행이 아니라 배치별 단계 요약값을 표본으로 사용해 의사 반복을 피했다.

In [ ]:
metrics = ['페니실린기울기', 'pH표준편차', '기질평균', 'OUR평균', 'CO2평균', 'DO평균', '온도표준편차']
results = []
for stage in stage_ranges:
    stage_data = stage_metrics.loc[stage_metrics['단계'].eq(stage)]
    for metric in metrics:
        groups = [stage_data.loc[stage_data['전략'].eq(strategy), metric].dropna().to_numpy()
                  for strategy in ['RC', 'OC', 'APC']]
        statistic, p_value, df2 = welch_anova(*groups)
        results.append({
            '단계': stage, '지표': metric,
            'RC평균': groups[0].mean(), 'OC평균': groups[1].mean(), 'APC평균': groups[2].mean(),
            'Welch_F': statistic, 'df2': df2, 'p값': p_value,
        })
stage_results = pd.DataFrame(results)
stage_results['FDR'] = fdr_bh(stage_results['p값'])
significant = stage_results.loc[stage_results['FDR'] < 0.05].sort_values('FDR')
print(f'FDR 유의: {len(significant)}/{len(stage_results)}개')
display(significant.round(6))

### 최종 판단

- 28개 검정 중 8개가 FDR 기준 유의했고, 차이는 초기·성장기보다 생산기와 후기에 집중됐다.
- 생산기에는 기질 평균과 페니실린 농도 기울기가 전략별로 달랐다. APC의 생산기 기울기(0.138)가 RC(0.105)와 OC(0.101)보다 높다.
- 후기 페니실린 기울기는 RC 0.008, OC -0.047, APC 0.074로 가장 큰 차이를 보였다. OC는 후기 농도가 감소하고 APC는 계속 증가하는 패턴이다.
- 후기 APC는 pH 변동(표준편차 약 0.0165)이 작고, 기질 평균(약 0.0014)이 거의 0이며, OUR와 CO2 평균은 상대적으로 높다. 후기 온도 표준편차도 전략별 차이가 확인됐다. 후기 운전 방식이 전략 성과 차이의 핵심 후보이다.
- 이 검정은 세 전략 중 적어도 하나가 다르다는 것만 말한다. 특정 두 전략의 차이를 확정하려면 단계·지표별 사후 쌍대검정과 추가 FDR 보정이 필요하다.
- 단계는 각 배치의 실제 종료시간으로 계산했으므로 종료 후 설명용이다. 실제 종료시간을 모르는 실시간 모델 입력으로 그대로 사용하면 미래정보 누출이 된다.